# Join FJC IDB data to CL Circuit data based on cleaned docket numbers

# Import libraries

In [1]:
import numpy as np
import pandas as pd

from fjc_utils import *
from cl_utils import *
from clean_utils import *

# Join FJC to CL Circuit on cleaned docket number

In [2]:
fjc_df = pd.read_csv("fjc_cleaned.csv")
len(fjc_df)

1989051

In [3]:
circuit_df = pd.read_csv("cl_circuits_clean.csv")
len(circuit_df)

1441430

In [4]:
# Find ["CIRCUIT_STR", "DOCKET_NUM"] combos in fjc_df not present in ["court_id", "docket_number"] in circuit_df
fjc_keys = fjc_df[["CIRCUIT_STR", "DOCKET_NUM"]].drop_duplicates()
circuit_keys = circuit_df[["court_id", "docket_number"]].drop_duplicates()

not_in_circuit = fjc_keys.merge(
    circuit_keys,
    left_on=["CIRCUIT_STR", "DOCKET_NUM"],
    right_on=["court_id", "docket_number"],
    how="left",
    indicator=True
).query('_merge == "left_only"')[["CIRCUIT_STR", "DOCKET_NUM"]]

not_in_circuit.head()

,CIRCUIT_STR,DOCKET_NUM
1,cadc,21080
5,cadc,21816
10,cadc,22210
12,cadc,22275
17,cadc,22495


In [5]:
len(not_in_circuit[["CIRCUIT_STR", "DOCKET_NUM"]].drop_duplicates())

1039026

In [6]:
merged_df = pd.merge(circuit_df, fjc_df, left_on=["court_id", "docket_number"], right_on=["CIRCUIT_STR", "DOCKET_NUM"], how="left")
len(merged_df)

1504694

## look at the records that did not match on docket number

In [7]:
no_match = merged_df[merged_df["DOCKET_NUM"].isna()]
len(no_match)

364668

In [8]:
len(no_match[["docket_number", "court_id"]].drop_duplicates())

283909

In [9]:
no_match.to_csv("circuits_no_docket_match.csv", index=False)

In [10]:
no_match.head()

,docket_id,docket_number_raw,cluster_id,case_name,date_filed,precedential_status,court_id,docket_type,docket_number,docket_format_confirmed,...,DOCKET_NUM,CIRCUIT_STR,DDIST_STR,APPTYPE_STR,DISP_STR,OUTCOME_STR,PROCTERM_STR,METHOD_STR,PUBSTAT_STR,DISP_UNIFIED
21,8651,19-13843,472932,United States v. Morrison,1986-06-26,Published,ca11,singles,19-13843,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,16762,12-14385,1086837,Henry Bravo Benitez v. US Attorney General,2013-10-24,Unpublished,ca11,singles,12-14385,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,23923,13-10316,2643736,Pedro Rafael Pereira Olivares v. U.S. Attorney...,2013-11-22,Unpublished,ca11,singles,13-10316,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,7364,19-13277,177999,United States v. Wayerski,2010-10-26,Published,ca11,singles,19-13277,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,34150,09-11903,41377,Fernando Demetrio Cadena Chunza v. US Atty. Gen.,2010-01-12,Unpublished,ca11,singles,09-11903,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## remove rows where there was no match on docket number

In [11]:
merged_df = merged_df[~merged_df["DOCKET_NUM"].isna()]
len(merged_df)

1140026

In [12]:
len(merged_df[["docket_number", "court_id"]].drop_duplicates())

890770

In [13]:
len(merged_df[["docket_number", "court_id"]].drop_duplicates()) / len(circuit_df[["docket_number", "court_id"]].drop_duplicates())

0.7583092913042627

In [14]:
len(merged_df[["docket_number", "court_id"]].drop_duplicates()) / len(fjc_df[["DOCKET_NUM", "CIRCUIT_STR"]].drop_duplicates())

0.46158764967903343

In [15]:
merged_df.to_csv("fjc_cl_circuit_merged.csv", index=False)